# Train and validate the exploratory BERT evaluator

Use a fresh GPU runtime. First human-label the blinded candidate responses locally,
then run the assembly command to produce `bert.jsonl`. Upload that file here. You may
also upload `generated_test.jsonl` for a separate real-output domain-shift check.
Do **not** upload final human labels as training data.

This is a tiny four-class baseline: moral, immoral, mixed, neutral. It can learn
shortcuts and source-specific writing styles. Human labels are primary evidence.
If held-out results are poor or classes are missing, report that and use BERT only as
an exploratory secondary measure; do not present it as an objective moral authority.

The train/validation/test assignment is fixed by scenario. Checkpoints are selected
only on validation macro-F1. The test sets are evaluated after training; do not tune
against them. No final-evaluation rows enter training or checkpoint selection.


In [ ]:
import sys,subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'transformers==4.51.3','datasets==3.5.0','accelerate==1.6.0','scikit-learn>=1.5,<2'])
print('Restart if requested, then continue below.')


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import json,hashlib,math,importlib.metadata
import numpy as np
import torch
from google.colab import files
from datasets import Dataset
from sklearn.metrics import classification_report,confusion_matrix,f1_score
from transformers import AutoTokenizer,AutoModelForSequenceClassification,Trainer,TrainingArguments,DataCollatorWithPadding,set_seed
assert torch.cuda.is_available(), 'Choose a GPU runtime.'
RUN=Path('/content/bert_evaluator_runs')/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN.mkdir(parents=True)
uploaded=files.upload()
assert 'bert.jsonl' in uploaded, 'Upload assembled bert.jsonl, optionally generated_test.jsonl.'
rows=[json.loads(line) for line in uploaded['bert.jsonl'].decode().splitlines() if line.strip()]
LABELS=['moral','immoral','mixed','neutral']
label2id={label:i for i,label in enumerate(LABELS)}
assert rows and all(r.get('reviewed') is True and r['label'] in LABELS and r['purpose'].startswith('evaluator_') and r['record_type']=='anchor' for r in rows)
parts={s:[r for r in rows if r['split']==s] for s in ['train','validation','test']}
assert all(parts.values()), 'All three fixed splits need human labels.'
assert len({r['id'] for r in rows})==len(rows)
for a,b in [('train','validation'),('train','test'),('validation','test')]:
    assert not {r['group_id'] for r in parts[a]} & {r['group_id'] for r in parts[b]}, 'Scenario leakage'
    assert not {r['prompt'].strip().casefold() for r in parts[a]} & {r['prompt'].strip().casefold() for r in parts[b]}, 'Prompt leakage'
for split,items in parts.items():
    counts=Counter(r['label'] for r in items);print(split,dict(counts))
    assert set(counts)==set(LABELS), f'{split}: missing classes; collect more labels instead of inventing labels.'
assert min(Counter(r['label'] for r in parts['train']).values())>=2
(RUN/'bert.jsonl').write_bytes(uploaded['bert.jsonl'])
MODEL='google-bert/bert-base-uncased'
REVISION='86b5e0934494bd15c9632b12f734a8a67f723594'
set_seed(42)
tokenizer=AutoTokenizer.from_pretrained(MODEL,revision=REVISION)

def make_dataset(items):
    for r in items:
        assert len(tokenizer(r['response'],add_special_tokens=False)['input_ids'])<=250, 'Long response needs explicit review; no silent answer truncation.'
    encoded=tokenizer([r['prompt'] for r in items],[r['response'] for r in items],
        max_length=512,truncation='only_first')
    return Dataset.from_dict(dict(encoded)|{'labels':[label2id[r['label']] for r in items]})

model=AutoModelForSequenceClassification.from_pretrained(MODEL,revision=REVISION,
    num_labels=4,id2label=dict(enumerate(LABELS)),label2id=label2id,use_safetensors=True)
def metrics(p):
    return {'macro_f1':f1_score(p.label_ids,p.predictions.argmax(-1),labels=list(range(4)),average='macro',zero_division=0)}
trainer=Trainer(model=model,args=TrainingArguments(output_dir=str(RUN/'checkpoints'),
    learning_rate=2e-5,per_device_train_batch_size=4,gradient_accumulation_steps=4,
    per_device_eval_batch_size=4,num_train_epochs=3,weight_decay=0.01,
    eval_strategy='epoch',save_strategy='epoch',load_best_model_at_end=True,
    metric_for_best_model='macro_f1',greater_is_better=True,save_total_limit=2,
    report_to='none',seed=42,data_seed=42),
    train_dataset=make_dataset(parts['train']),eval_dataset=make_dataset(parts['validation']),
    data_collator=DataCollatorWithPadding(tokenizer),compute_metrics=metrics)
training_result=trainer.train()
assert math.isfinite(training_result.training_loss), 'Non-finite BERT training loss'
trainer.save_model(str(RUN/'evaluator'));tokenizer.save_pretrained(RUN/'evaluator')


In [ ]:
def report(items,name):
    pred=trainer.predict(make_dataset(items))
    predicted=pred.predictions.argmax(-1);gold=pred.label_ids
    result={'rows':len(items),'scenario_count':len({r['group_id'] for r in items}),
        'classification_report':classification_report(gold,predicted,labels=list(range(4)),target_names=LABELS,output_dict=True,zero_division=0),
        'confusion_matrix':confusion_matrix(gold,predicted,labels=list(range(4))).tolist(),
        'label_order':LABELS,'notes':'Point estimates on a small exploratory test set; human labels required.'}
    (RUN/(name+'_report.json')).write_text(json.dumps(result,indent=2))
    print(name,classification_report(gold,predicted,labels=list(range(4)),target_names=LABELS,zero_division=0))
    return result

anchor_test=report(parts['test'],'heldout_anchor_test')
if 'generated_test.jsonl' in uploaded:
    actual=[json.loads(s) for s in uploaded['generated_test.jsonl'].decode().splitlines() if s.strip()]
    allowed={r['group_id'] for r in parts['test']}
    assert actual and all(r['split']=='test' and r['purpose']=='evaluator_test' and r['record_type']=='generated' and r['reviewed'] is True and r['group_id'] in allowed for r in actual)
    usable=[r for r in actual if r['label'] in LABELS and len(tokenizer(r['response'],add_special_tokens=False)['input_ids'])<=250]
    (RUN/'generated_test_coverage.json').write_text(json.dumps({'total':len(actual),'scorable':len(usable),'excluded':len(actual)-len(usable)}))
    if usable:report(usable,'generated_output_domain_test')
manifest={'model':MODEL,'revision':REVISION,'seed':42,'label_order':LABELS,
    'dataset_sha256':hashlib.sha256(uploaded['bert.jsonl']).hexdigest(),
    'counts':{s:len(r) for s,r in parts.items()},'best_checkpoint':trainer.state.best_model_checkpoint,
    'versions':{p:importlib.metadata.version(p) for p in ['torch','transformers','datasets','accelerate','scikit-learn']},
    'limitations':'Tiny, source-mixed evaluator with human-verified labels. Low held-out scores or absent domain classes make automated moral conclusions unreliable. Do not tune on final evaluation.'}
(RUN/'bert_manifest.json').write_text(json.dumps(manifest,indent=2))


## Download evaluator and metrics

Save before ending the runtime. Share the reports so we can assess whether BERT is useful. Human judgments remain primary.

In [ ]:
import shutil
files.download(shutil.make_archive(str(RUN),'zip',root_dir=RUN))
